# Treinamento de Modelos Deep Learning Darts

Foco em redes neurais avançadas para séries temporais.

### Importação das Bibliotecas
Framework principal: `darts` para a modelagem via PyTorch.

In [ ]:
import pandas as pd
import numpy as np
import warnings
import os
import joblib
import logging
import torch
from sklearn.model_selection import TimeSeriesSplit
from darts import TimeSeries
from darts.models import BlockRNNModel, TCNModel
from darts.dataprocessing.transformers import Scaler
import optuna

warnings.filterwarnings('ignore')
logging.getLogger("pytorch_lightning.utilities.rank_zero").setLevel(logging.WARNING)
logging.getLogger("pytorch_lightning.accelerators.cuda").setLevel(logging.WARNING)
optuna.logging.set_verbosity(optuna.logging.WARNING)
os.makedirs('models/result_pkl', exist_ok=True)
os.makedirs('resultados', exist_ok=True)

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2))

def evaluate(y_true, y_pred):
    y_true, y_pred = np.array(y_true).flatten(), np.array(y_pred).flatten()
    mae = np.mean(np.abs(y_true - y_pred))
    r = rmse(y_true, y_pred)
    mean_y = np.mean(y_true)
    nrmse = r / mean_y if mean_y != 0 else 0
    smape = 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8))
    return {'MAE': round(mae, 2), 'RMSE': round(r, 2), 'nRMSE': round(nrmse, 4), 'sMAPE': round(smape, 2)}

### Preparação no Padrão Darts
Diferente dos modelos ML, aqui retornamos o DataFrame completo estruturado para ser convertido no formato nativo `TimeSeries` no momento do treino.

In [ ]:
def load_data_darts(csv_path):
    df = pd.read_csv(csv_path, parse_dates=['Date']).sort_values('Date').reset_index(drop=True)
    target = 'Revenue'
    unavailable_cols = {'UnitPrice_median', 'UnitPrice_std', 'UniqueCustomers', 'UniqueProducts'}
    feature_cols = [
        c for c in df.columns
        if c not in ['Date', target]
        and pd.api.types.is_numeric_dtype(df[c])
        and not c.startswith(('RevenueLag_', 'Rolling'))
        and c not in unavailable_cols
    ]

    cutoff = pd.Timestamp('2011-09-01')
    train_df = df[df['Date'] < cutoff].reset_index(drop=True)
    test_df = df[df['Date'] >= cutoff].reset_index(drop=True)

    return train_df, test_df, feature_cols


def prepare_fold_features(train_df, apply_df):
    frame = apply_df.copy()

    overall_mean = train_df['Revenue'].mean()
    overall_median = train_df['Revenue'].median()
    overall_q25 = train_df['Revenue'].quantile(0.25)
    overall_q75 = train_df['Revenue'].quantile(0.75)
    overall_iqr = overall_q75 - overall_q25
    overall_std = train_df['Revenue'].std()
    if pd.isna(overall_std):
        overall_std = 0.0

    group_defaults = {
        'Weekday': {'mean': overall_mean, 'median': overall_median, 'IQR': overall_iqr},
        'Month': {'mean': overall_mean, 'median': overall_median, 'IQR': overall_iqr},
        'Quarter': {'mean': overall_mean, 'median': overall_median, 'IQR': overall_iqr},
    }

    for col, defaults in group_defaults.items():
        stats = train_df.groupby(col)['Revenue'].agg(
            mean='mean',
            median='median',
            q25=lambda x: x.quantile(0.25),
            q75=lambda x: x.quantile(0.75),
        )
        stats['IQR'] = stats['q75'] - stats['q25']
        frame[f'{col}Revenue_mean'] = frame[col].map(stats['mean']).fillna(defaults['mean'])
        frame[f'{col}Revenue_median'] = frame[col].map(stats['median']).fillna(defaults['median'])
        frame[f'{col}Revenue_IQR'] = frame[col].map(stats['IQR']).fillna(defaults['IQR'])

    xmas_stats = train_df.groupby('PreChristmas')['Revenue'].agg(
        PreChristmasRevenue_mean='mean',
        PreChristmasRevenue_std='std'
    )
    frame['PreChristmasRevenue_mean'] = frame['PreChristmas'].map(xmas_stats['PreChristmasRevenue_mean']).fillna(overall_mean)
    frame['PreChristmasRevenue_std'] = frame['PreChristmas'].map(xmas_stats['PreChristmasRevenue_std']).fillna(overall_std)
    frame['PreChristmasRevenue_std'] = frame['PreChristmasRevenue_std'].fillna(0.0)

    return frame

## 2. Carregamento dos Dados

### Carregamento das Partições

In [ ]:
datasets = {
    'Produto': 'data/product.csv',
    'País':    'data/country.csv',
    'Cliente': 'data/customer.csv'
}

data = {}
for name, path in datasets.items():
    train_df, test_df, feat_cols = load_data_darts(path)
    data[name] = {'train_df': train_df, 'test_df': test_df, 'feature_cols': feat_cols}
    print(f"  {name}: train_samples={len(train_df)}, test_samples={len(test_df)}")

## 3. RNN (LSTM) via Darts (5-Fold Time Series Split)

### Estruturação e Escalonamento do BlockRNN (LSTM)
Em cada fold temporal, aplica-se o `Scaler` para deixar as features na mesma magnitude.
Crucial: As covariáveis (`past_covariates`) do treino e validação são concatenadas (`cov_full_scaled`) para alimentar as etapas de projeção no futuro.

**Correção aplicada:**
- `MAE_Test` / `RMSE_Test`: calculados sobre o conjunto de **teste real** (futuro), após treinar o modelo final com todo o conjunto de treino.
- `MAE_CV` / `RMSE_CV`: mantidos como diagnóstico de estabilidade temporal.

In [ ]:
N_SPLITS = 5
tscv = TimeSeriesSplit(n_splits=N_SPLITS)
pl_trainer_kwargs = {"enable_progress_bar": False}

for name in data.keys():
    data[name]['train_df']['Revenue_Log'] = np.log1p(data[name]['train_df']['Revenue'])
    data[name]['test_df']['Revenue_Log'] = np.log1p(data[name]['test_df']['Revenue'])

results_lstm = {}
for name, d in data.items():
    print(f"\nTreinando LSTM para a serie: {name}")
    train_df, test_df, feature_cols = d['train_df'], d['test_df'], d['feature_cols']

    def objective_lstm(trial):
        hidden_dim = trial.suggest_categorical('hidden_dim', [16, 32])
        dropout = trial.suggest_float('dropout', 0.1, 0.3)
        lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)

        fold_maes = []
        fold_rmses = []
        for train_idx, val_idx in tscv.split(train_df):
            fold_train_df = train_df.iloc[train_idx].copy()
            fold_val_df = train_df.iloc[val_idx].copy()

            fold_train_features = prepare_fold_features(fold_train_df, fold_train_df)
            fold_val_features = prepare_fold_features(fold_train_df, fold_val_df)

            tgt_train = TimeSeries.from_dataframe(fold_train_features, 'Date', 'Revenue_Log')
            cov_train = TimeSeries.from_dataframe(fold_train_features, 'Date', feature_cols)
            tgt_val = TimeSeries.from_dataframe(fold_val_features, 'Date', 'Revenue_Log')
            cov_val = TimeSeries.from_dataframe(fold_val_features, 'Date', feature_cols)

            scaler_tgt = Scaler()
            scaler_cov = Scaler()

            tgt_train_scaled = scaler_tgt.fit_transform(tgt_train)
            cov_train_scaled = scaler_cov.fit_transform(cov_train)
            cov_val_scaled = scaler_cov.transform(cov_val)
            cov_full_scaled = cov_train_scaled.append(cov_val_scaled)

            model = BlockRNNModel(
                model='LSTM', hidden_dim=hidden_dim, n_rnn_layers=1, dropout=dropout,
                batch_size=16, n_epochs=15, optimizer_kwargs={'lr': lr},
                input_chunk_length=7, output_chunk_length=1, random_state=42,
                pl_trainer_kwargs=pl_trainer_kwargs
            )

            if len(tgt_train_scaled) > 7:
                model.fit(tgt_train_scaled, past_covariates=cov_train_scaled)
                pred_scaled = model.predict(n=len(tgt_val), series=tgt_train_scaled, past_covariates=cov_full_scaled)
                pred_log = scaler_tgt.inverse_transform(pred_scaled)

                preds = np.expm1(pred_log.values())
                y_val = np.expm1(tgt_val.values())
                fold_maes.append(np.mean(np.abs(y_val - preds)))
                fold_rmses.append(rmse(y_val, preds))

        mean_mae = float(np.mean(fold_maes)) if fold_maes else float('inf')
        mean_rmse = float(np.mean(fold_rmses)) if fold_rmses else float('inf')
        trial.set_user_attr('mae_mean', mean_mae)
        trial.set_user_attr('rmse_mean', mean_rmse)
        return mean_rmse

    study = optuna.create_study(direction='minimize')
    study.optimize(objective_lstm, n_trials=5) # Reduzido para limite computacional

    best_trial = study.best_trial
    best_params = best_trial.params
    full_train_features = prepare_fold_features(train_df, train_df)
    test_features = prepare_fold_features(train_df, test_df)

    tgt_full_train = TimeSeries.from_dataframe(full_train_features, 'Date', 'Revenue_Log')
    cov_full_train = TimeSeries.from_dataframe(full_train_features, 'Date', feature_cols)
    tgt_test = TimeSeries.from_dataframe(test_features, 'Date', 'Revenue_Log')
    cov_test = TimeSeries.from_dataframe(test_features, 'Date', feature_cols)

    scaler_tgt_final = Scaler()
    scaler_cov_final = Scaler()

    tgt_full_train_scaled = scaler_tgt_final.fit_transform(tgt_full_train)
    cov_full_train_scaled = scaler_cov_final.fit_transform(cov_full_train)
    cov_test_scaled = scaler_cov_final.transform(cov_test)
    cov_all_scaled = cov_full_train_scaled.append(cov_test_scaled)

    final_model = BlockRNNModel(
        model='LSTM', hidden_dim=best_params['hidden_dim'], n_rnn_layers=1, dropout=best_params['dropout'],
        batch_size=16, n_epochs=20, optimizer_kwargs={'lr': best_params['lr']},
        input_chunk_length=7, output_chunk_length=1, random_state=42,
        pl_trainer_kwargs=pl_trainer_kwargs
    )
    final_model.fit(tgt_full_train_scaled, past_covariates=cov_full_train_scaled)
    pred_test_scaled = final_model.predict(n=len(tgt_test), series=tgt_full_train_scaled, past_covariates=cov_all_scaled)

    pred_test_log = scaler_tgt_final.inverse_transform(pred_test_scaled)
    test_preds = np.expm1(pred_test_log.values())
    y_test_real = np.expm1(tgt_test.values())

    test_metrics = evaluate(y_test_real, test_preds)
    final_model.save(f'models/result_pkl/lstm_{name}.pt')

    results_lstm[name] = {
        'MAE_CV': round(best_trial.user_attrs['mae_mean'], 2),
        'RMSE_CV': round(best_trial.user_attrs['rmse_mean'], 2),
        'MAE_Test': test_metrics['MAE'],
        'RMSE_Test': test_metrics['RMSE'],
        'nRMSE_Test': test_metrics['nRMSE'],
        'sMAPE_Test': test_metrics['sMAPE'],
        'periodos_previstos': len(tgt_test),
        'valores_previstos': test_preds.flatten().tolist()
    }

## 4. TCN via Darts (5-Fold Time Series Split)

### Treinamento de Redes Convolucionais Temporais (TCN)
Baseando-se em convoluções causais dilatadas, avalia janelas de tempo completas com alto poder de extração de picos locais.

**Correção aplicada:** mesma do LSTM — `MAE_Test`/`RMSE_Test` calculados no conjunto de teste real.

In [ ]:
results_tcn = {}
for name, d in data.items():
    print(f"\nTreinando TCN para a serie: {name}")
    train_df, test_df, feature_cols = d['train_df'], d['test_df'], d['feature_cols']

    def objective_tcn(trial):
        dropout = trial.suggest_float('dropout', 0.1, 0.3)
        lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)

        fold_maes = []
        fold_rmses = []
        for train_idx, val_idx in tscv.split(train_df):
            fold_train_df = train_df.iloc[train_idx].copy()
            fold_val_df = train_df.iloc[val_idx].copy()

            fold_train_features = prepare_fold_features(fold_train_df, fold_train_df)
            fold_val_features = prepare_fold_features(fold_train_df, fold_val_df)

            tgt_train = TimeSeries.from_dataframe(fold_train_features, 'Date', 'Revenue_Log')
            cov_train = TimeSeries.from_dataframe(fold_train_features, 'Date', feature_cols)
            tgt_val = TimeSeries.from_dataframe(fold_val_features, 'Date', 'Revenue_Log')
            cov_val = TimeSeries.from_dataframe(fold_val_features, 'Date', feature_cols)

            scaler_tgt, scaler_cov = Scaler(), Scaler()
            tgt_train_scaled = scaler_tgt.fit_transform(tgt_train)
            cov_train_scaled = scaler_cov.fit_transform(cov_train)
            cov_val_scaled = scaler_cov.transform(cov_val)
            cov_full_scaled = cov_train_scaled.append(cov_val_scaled)

            model = TCNModel(
                input_chunk_length=7, output_chunk_length=1, dropout=dropout,
                batch_size=16, n_epochs=15, optimizer_kwargs={'lr': lr},
                random_state=42, pl_trainer_kwargs=pl_trainer_kwargs
            )

            if len(tgt_train_scaled) > 7:
                model.fit(tgt_train_scaled, past_covariates=cov_train_scaled)
                pred_scaled = model.predict(n=len(tgt_val), series=tgt_train_scaled, past_covariates=cov_full_scaled)
                pred_log = scaler_tgt.inverse_transform(pred_scaled)

                preds = np.expm1(pred_log.values())
                y_val = np.expm1(tgt_val.values())
                fold_maes.append(np.mean(np.abs(y_val - preds)))
                fold_rmses.append(rmse(y_val, preds))

        mean_mae = float(np.mean(fold_maes)) if fold_maes else float('inf')
        mean_rmse = float(np.mean(fold_rmses)) if fold_rmses else float('inf')
        trial.set_user_attr('mae_mean', mean_mae)
        trial.set_user_attr('rmse_mean', mean_rmse)
        return mean_rmse

    study = optuna.create_study(direction='minimize')
    study.optimize(objective_tcn, n_trials=5)

    best_trial = study.best_trial
    best_params = best_trial.params
    full_train_features = prepare_fold_features(train_df, train_df)
    test_features = prepare_fold_features(train_df, test_df)

    tgt_full_train = TimeSeries.from_dataframe(full_train_features, 'Date', 'Revenue_Log')
    cov_full_train = TimeSeries.from_dataframe(full_train_features, 'Date', feature_cols)
    tgt_test = TimeSeries.from_dataframe(test_features, 'Date', 'Revenue_Log')
    cov_test = TimeSeries.from_dataframe(test_features, 'Date', feature_cols)

    scaler_tgt_final, scaler_cov_final = Scaler(), Scaler()
    tgt_full_train_scaled = scaler_tgt_final.fit_transform(tgt_full_train)
    cov_full_train_scaled = scaler_cov_final.fit_transform(cov_full_train)
    cov_test_scaled = scaler_cov_final.transform(cov_test)
    cov_all_scaled = cov_full_train_scaled.append(cov_test_scaled)

    final_model = TCNModel(
        input_chunk_length=7, output_chunk_length=1, dropout=best_params['dropout'],
        batch_size=16, n_epochs=20, optimizer_kwargs={'lr': best_params['lr']},
        random_state=42, pl_trainer_kwargs=pl_trainer_kwargs
    )
    final_model.fit(tgt_full_train_scaled, past_covariates=cov_full_train_scaled)
    pred_test_scaled = final_model.predict(n=len(tgt_test), series=tgt_full_train_scaled, past_covariates=cov_all_scaled)

    pred_test_log = scaler_tgt_final.inverse_transform(pred_test_scaled)
    test_preds = np.expm1(pred_test_log.values())
    y_test_real = np.expm1(tgt_test.values())

    test_metrics = evaluate(y_test_real, test_preds)
    final_model.save(f'models/result_pkl/tcn_{name}.pt')

    results_tcn[name] = {
        'MAE_CV': round(best_trial.user_attrs['mae_mean'], 2),
        'RMSE_CV': round(best_trial.user_attrs['rmse_mean'], 2),
        'MAE_Test': test_metrics['MAE'],
        'RMSE_Test': test_metrics['RMSE'],
        'nRMSE_Test': test_metrics['nRMSE'],
        'sMAPE_Test': test_metrics['sMAPE'],
        'periodos_previstos': len(tgt_test),
        'valores_previstos': test_preds.flatten().tolist()
    }

### Exportação dos Resultados
Métricas de CV (diagnóstico) e de **teste real** (desempenho no futuro) persistidas em CSV.

In [ ]:
import os
os.makedirs('resultados', exist_ok=True)

rows = []
for name, m in results_lstm.items():
    rows.append({
        'serie':              name,
        'periodos_previstos': m['periodos_previstos'],
        'modelo':             'LSTM',
        'valores_previstos':  m['valores_previstos'],
        'MAE_CV':             round(m['MAE_CV'],    2),
        'RMSE_CV':            round(m['RMSE_CV'],   2),
        'MAE_Test':           round(m['MAE_Test'],  2),
        'RMSE_Test':          round(m['RMSE_Test'], 2),
    })
for name, m in results_tcn.items():
    rows.append({
        'serie':              name,
        'periodos_previstos': m['periodos_previstos'],
        'modelo':             'TCN',
        'valores_previstos':  m['valores_previstos'],
        'MAE_CV':             round(m['MAE_CV'],    2),
        'RMSE_CV':            round(m['RMSE_CV'],   2),
        'MAE_Test':           round(m['MAE_Test'],  2),
        'RMSE_Test':          round(m['RMSE_Test'], 2),
    })

df_results = pd.DataFrame(rows)
df_results.to_csv('resultados/resultados_DL.csv', index=False)
print('Resultados salvos em resultados/resultados_DL.csv')
df_results